# 오피스텔 RAG 추천 시스템

이 노트북은 `cleaned_officetel_data.csv`를 로드하고 전처리 및 RAG 기반 검색 시스템을 구축하는 과정을 단계별로 보여줍니다.

In [44]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from datetime import datetime
from scipy import stats
import os

## 1. 데이터 로드 및 전처리 함수 정의

In [45]:
def load_and_preprocess_data(file_path='cleaned_officetel_data.csv'):
    print("데이터 로드 중...")
    df = pd.read_csv(file_path)
    print(f"로드 완료: {len(df)}개 매물, {df.shape[1]}개 컬럼")
    
    # 가격 데이터 정규화
    df = normalize_price(df)
    
    # 임베딩 텍스트 생성
    print("임베딩 텍스트 생성 중...")
    df['embedding_text'] = df.apply(create_embedding_text, axis=1)
    
    # 샘플 텍스트 출력
    print("\n생성된 텍스트 샘플:")
    for i, text in enumerate(df['embedding_text'].head(3)):
        print(f"{i+1}. {text}")
    
    return df

## 2. 가격 데이터 정규화 함수 정의

In [46]:
def normalize_price(df):
    print("가격 데이터 정규화 중...")
    price_columns = ['월세평균', '평균전세가격', '월세보증금평균']
    for col in price_columns:
        if col in df.columns:
            # 이상치 제거 전 NaN 처리
            if df[col].isnull().sum() > 0:
                print(f"  - {col}: {df[col].isnull().sum()}개 결측치")
            
            # 숫자형으로 변환
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
            # 이상치 확인 (NaN 제외)
            valid_data = df[col].dropna()
            if len(valid_data) > 0:
                z_scores = stats.zscore(valid_data)
                abs_z_scores = np.abs(z_scores)
                outlier_mask = abs_z_scores > 3
                outliers_count = outlier_mask.sum()
                
                if outliers_count > 0:
                    print(f"  - {col} 이상치 발견: {outliers_count}개")
                
                # 단위 변환 (원 → 만원)
                df[f'{col}_만원'] = df[col] / 10000
                print(f"  - {col} 변환 완료: 평균 {df[f'{col}_만원'].mean():.2f}만원")
    
    return df

## 3. 임베딩 텍스트 생성 함수 정의

In [47]:
def create_embedding_text(row):
    """오피스텔 정보를 자연어 텍스트로 변환"""
    text = ""
    
    # 기본 정보
    if pd.notna(row.get('건물이름')):
        text += f"{row['건물이름']}은(는) "
    
    if pd.notna(row.get('지번주소')):
        text += f"{row['지번주소']}에 위치한 오피스텔입니다. "
    
    # 면적 정보
    if pd.notna(row.get('최근거래전용면적')):
        try:
            area_val = float(row['최근거래전용면적'])
            pyeong = area_val / 3.3058
            text += f"전용면적은 {area_val:.1f}㎡({pyeong:.1f}평)입니다. "
        except (ValueError, TypeError):
            pass
    
    # 임대 정보
    if pd.notna(row.get('월세평균')) and row['월세평균'] > 0:
        try:
            deposit = float(row.get('월세보증금평균', 0))
            monthly = float(row['월세평균'])
            text += f"보증금 {int(deposit/10000)}만원에 월세는 {int(monthly/10000)}만원입니다. "
        except (ValueError, TypeError):
            pass
    elif pd.notna(row.get('평균전세가격')) and row['평균전세가격'] > 0:
        try:
            jeonse = float(row['평균전세가격'])
            text += f"전세가는 {int(jeonse/10000)}만원입니다. "
        except (ValueError, TypeError):
            pass
    else:
        text += "임대 조건 정보가 부족합니다. "
    
    # 추가 정보
    if pd.notna(row.get('근처지하철역')):
        text += f"가까운 지하철역은 {row['근처지하철역']}입니다. "
    
    if pd.notna(row.get('준공연도')):
        try:
            current_year = datetime.now().year
            building_year = int(float(row['준공연도']))
            building_age = current_year - building_year
            text += f"{building_year}년에 준공되어 건물 연식은 {building_age}년입니다. "
        except (ValueError, TypeError):
            pass
    
    return text.strip()

## 4. RAG용 임베딩 및 FAISS 인덱스 생성 함수 정의

In [48]:
def create_embeddings_and_index(df, model_name='jhgan/ko-sroberta-multitask', force_recreate=False):
    vector_dir = 'vectorData\\'
    embedding_path = os.path.join(vector_dir, 'embedding_texts_rag.csv')
    index_path = os.path.join(vector_dir, 'officetel_rag_faiss.index')
    
    # 저장된 임베딩이 있는지 확인
    if os.path.exists(embedding_path) and os.path.exists(index_path) and not force_recreate:
        print("저장된 임베딩과 인덱스를 로드합니다...")
        embedding_df = pd.read_csv(embedding_path)
        
        # FAISS 인덱스 로드
        index = faiss.read_index(index_path)
        print(f"로드 완료: {index.ntotal}개 벡터, 차원 {index.d}")
        
        return embedding_df, index
    
    # 새로 임베딩 생성
    print(f"임베딩 모델 '{model_name}' 로드 중...")
    model = SentenceTransformer(model_name)
    
    print("텍스트 임베딩 생성 중...")
    texts = df['embedding_text'].tolist()
    embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    
    # FAISS 인덱스 생성
    print("FAISS 인덱스 생성 중...")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)  # 내적 기반 (코사인 유사도용)
    index.add(embeddings)
    
    print(f"인덱스 생성 완료: {index.ntotal}개 벡터, 차원 {dimension}")
    
    # 디렉토리가 없으면 생성
    os.makedirs(vector_dir, exist_ok=True)
    
    # 임베딩 및 인덱스 저장
    print("임베딩 및 인덱스 저장 중...")
    df.to_csv(embedding_path, index=False)
    faiss.write_index(index, index_path)
    print("저장 완료")
    
    return df, index


## 5. 검색 및 추천 함수 정의

In [49]:
def search_similar_properties(query, df, index, model_name='jhgan/ko-sroberta-multitask', top_k=5):
    print(f"쿼리: '{query}' 검색 중...")
    
    # 모델 로드
    model = SentenceTransformer(model_name)
    
    # 쿼리 임베딩
    query_embedding = model.encode([query], normalize_embeddings=True)
    
    # FAISS 검색
    distances, indices = index.search(query_embedding, top_k)
    
    # 결과 매핑
    results = df.iloc[indices[0]].copy()
    results['similarity'] = distances[0]
    
    print(f"검색 완료: {len(results)}개 결과")
    return results

# 6. 추천

In [50]:
def print_recommendations(results):
    print("\n 추천 오피스텔:")
    print("=" * 80)
    
    for i, (idx, row) in enumerate(results.iterrows()):
        print(f"[매물 {i+1}] 유사도: {row.get('similarity', 0):.4f}")
        print(f"이름: {row.get('건물이름', '정보 없음')}")
        print(f"위치: {row.get('지번주소', '정보 없음')}")
        
        # 면적 정보
        if pd.notna(row.get('최근거래전용면적')):
            pyeong = row['최근거래전용면적'] / 3.3058
            print(f"전용면적: {row['최근거래전용면적']:.1f}㎡({pyeong:.1f}평)")
        
        # 가격 정보
        if pd.notna(row.get('월세평균')) and row['월세평균'] > 0:
            deposit = row.get('월세보증금평균', 0)
            monthly = row['월세평균']
            print(f"월세: 보증금 {int(deposit/10000)}만원 / 월세 {int(monthly/10000)}만원")
        
        if pd.notna(row.get('평균전세가격')) and row['평균전세가격'] > 0:
            jeonse = row['평균전세가격']
            print(f"전세: {int(jeonse/10000)}만원")
        
        # 추가 정보
        if pd.notna(row.get('근처지하철역')):
            print(f"근처 지하철: {row['근처지하철역']}")
        
        if pd.notna(row.get('준공연도')):
            current_year = datetime.now().year
            building_year = int(float(row['준공연도']))
            print(f"준공연도: {building_year}년 (건물 연식: {current_year - building_year}년)")
        
        print("-" * 80)

## 6. 필터링 함수 정의

In [ ]:
def filter_recommendations(results, max_monthly=None, min_area=None, location=None):
    filtered = results.copy()
    
    # 월세 필터링
    if max_monthly is not None:
        filtered = filtered[
            (filtered['월세평균'].notna() & (filtered['월세평균'] <= max_monthly * 10000)) |
            (filtered['월세평균'].isna())
        ]
    
    # 면적 필터링
    if min_area is not None:
        filtered = filtered[
            (filtered['공급면적'].notna() & (filtered['공급면적'] >= min_area)) |
            (filtered['공급면적'].isna())
        ]
    
    # 위치 필터링
    if location is not None:
        filtered = filtered[filtered['지번주소'].str.contains(location, na=False)]
    
    print(f"필터링 결과: {len(filtered)}개 매물")
    return filtered

## 7. Main 함수 및 인터랙티브 실행

In [54]:
def main():
    try:
        df = load_and_preprocess_data(r'C:\Users\user\Desktop\Final_Project\data\cleaned_officetel_data.csv')
    except FileNotFoundError:
        print("파일을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
        print("테스트용 데이터를 생성합니다...")
        df = pd.read_csv(r'C:\Users\user\Desktop\Final_Project\data\data\zu_officetels.csv')  
        df.to_csv(r'C:\Users\user\Desktop\Final_Project\data\cleaned_officetel_data.csv', index=False)
        df = load_and_preprocess_data(r'C:\Users\user\Desktop\Final_Project\data\data\zu_officetels.csv')
    
    # 임베딩 및 인덱스 생성
    df, index = create_embeddings_and_index(df)
    
    # 대화형 검색 인터페이스
    print("오피스텔 추천 시스템에 오신 것을 환영합니다!")
    print("원하시는 조건을 자연어로 입력해주세요. (종료: q)")
    
    while True:
        query = input("\n원하시는 오피스텔 조건을 입력해주세요: ")
        if query.lower() == 'q':
            print("검색을 종료합니다. 감사합니다!")
            break
        
        # 검색 실행
        results = search_similar_properties(query, df, index, top_k=10)
        
        # 필터링 옵션 입력
        print("\n추가 필터링 옵션 (없으면 엔터)")
        max_price_input = input("최대 월세 (만원): ")
        min_area_input = input("최소 면적 (㎡): ")
        location_input = input("선호 지역: ")
        
        max_price = float(max_price_input) if max_price_input else None
        min_area = float(min_area_input) if min_area_input else None
        location = location_input if location_input else None
        
        if max_price is not None or min_area is not None or location is not None:
            filtered_results = filter_recommendations(results, max_price, min_area, location)
            if len(filtered_results) == 0:
                print("필터링 결과가 없어 원래 검색 결과를 표시합니다.")
                print_recommendations(results)
            else:
                print_recommendations(filtered_results)
        else:
            print_recommendations(results)

if __name__ == "__main__":
    main()

데이터 로드 중...
로드 완료: 4165개 매물, 25개 컬럼
가격 데이터 정규화 중...
  - 월세평균: 860개 결측치
  - 월세평균 이상치 발견: 46개
  - 월세평균 변환 완료: 평균 63.99만원
  - 평균전세가격: 839개 결측치
  - 평균전세가격 이상치 발견: 25개
  - 평균전세가격 변환 완료: 평균 22240.47만원
  - 월세보증금평균: 860개 결측치
  - 월세보증금평균 이상치 발견: 72개
  - 월세보증금평균 변환 완료: 평균 5556.37만원
임베딩 텍스트 생성 중...

생성된 텍스트 샘플:
1. 유림은(는) 서울특별시 구로구 구로동1129-9에 위치한 오피스텔입니다. 전용면적은 19.0㎡(5.8평)입니다. 보증금 6883만원에 월세는 14만원입니다. 가까운 지하철역은 구로디지털단지역입니다. 2006년에 준공되어 건물 연식은 19년입니다.
2. DUO302	은(는) 서울특별시 중구 황학동819에 위치한 오피스텔입니다. 전용면적은 25.1㎡(7.6평)입니다. 보증금 3060만원에 월세는 58만원입니다. 가까운 지하철역은 신당역입니다. 2014년에 준공되어 건물 연식은 11년입니다.
3. 강남 푸르지오시티 2차(PRUGIO CITYⅡ)	은(는) 서울특별시 강남구 자곡동655에 위치한 오피스텔입니다. 보증금 3407만원에 월세는 57만원입니다. 가까운 지하철역은 수서역입니다. 2014년에 준공되어 건물 연식은 11년입니다.
저장된 임베딩과 인덱스를 로드합니다...
로드 완료: 4165개 벡터, 차원 768
오피스텔 추천 시스템에 오신 것을 환영합니다!
원하시는 조건을 자연어로 입력해주세요. (종료: q)
쿼리: '영등포구 전세 1억 매물 찾는중' 검색 중...
검색 완료: 10개 결과

추가 필터링 옵션 (없으면 엔터)


ValueError: could not convert string to float: '2억'